# 01_eda

Titanic EDA and missingness review.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib
base_dir = Path.cwd()
if base_dir.name != 'analytics':
    base_dir = base_dir / 'analytics'
mpl_config_dir = base_dir / '.matplotlib'
mpl_config_dir.mkdir(exist_ok=True, parents=True)
os.environ.setdefault('MPLCONFIGDIR', str(mpl_config_dir))
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Working directory:', base_dir)
df = sns.load_dataset('titanic')
print('Shape:', df.shape)
df.info()
display(df.describe(include='all'))
df.to_csv(base_dir / 'titanic.csv', index=False)
print('Saved Titanic fallback to', base_dir / 'titanic.csv')


In [ ]:
missing = (
    df.isnull()
      .mean()
      .mul(100)
      .loc[lambda s: s > 0]
      .sort_values(ascending=False)
)
print(missing.round(3))
display(missing.round(3))


### Missing-value decisions
- `age`: approximately 20% missing, so median imputation is appropriate because it falls in the medium-missingness band.
- `embarked` and `embark_town`: very low missingness, so mode imputation is sufficient.
- `deck` and `cabin`: extremely sparse, so they should be dropped rather than incorrectly imputed because the data are too incomplete for a defensible estimate.


In [ ]:
clean_df = df.copy()
clean_df['age'] = clean_df['age'].fillna(clean_df['age'].median())
clean_df['embarked'] = clean_df['embarked'].fillna(clean_df['embarked'].mode()[0])
clean_df['embark_town'] = clean_df['embark_town'].fillna(clean_df['embark_town'].mode()[0])
clean_df = clean_df.drop(columns=['deck', 'cabin'], errors='ignore')
print(clean_df.isnull().sum().loc[lambda s: s > 0])


In [ ]:
out_dir = base_dir / 'outputs'
out_dir.mkdir(exist_ok=True, parents=True)

plt.figure(figsize=(8, 5))
plt.hist(clean_df['age'], bins=30, color='#7aa6c2', edgecolor='black')
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig(out_dir / 'age_histogram.png', dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
plt.boxplot(clean_df['age'].dropna(), patch_artist=True, boxprops={'facecolor': '#7aa6c2'})
plt.title('Age Box Plot')
plt.ylabel('Age')
plt.tight_layout()
plt.savefig(out_dir / 'age_boxplot.png', dpi=150)
plt.close()

Q1 = clean_df['age'].quantile(0.25)
Q3 = clean_df['age'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
age_outliers = clean_df[(clean_df['age'] < lower) | (clean_df['age'] > upper)]
print('Age outliers:', len(age_outliers))


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(clean_df['fare'], bins=30, color='#7aa6c2', edgecolor='black')
plt.title('Fare Distribution')
plt.xlabel('Fare')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig(out_dir / 'fare_histogram.png', dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
plt.boxplot(clean_df['fare'].dropna(), patch_artist=True, boxprops={'facecolor': '#7aa6c2'})
plt.title('Fare Box Plot')
plt.ylabel('Fare')
plt.tight_layout()
plt.savefig(out_dir / 'fare_boxplot.png', dpi=150)
plt.close()

fare_mean = clean_df['fare'].mean()
fare_median = clean_df['fare'].median()
fare_mode = clean_df['fare'].mode()[0]
print('Mean:', fare_mean)
print('Median:', fare_median)
print('Mode:', fare_mode)
print('Skewness reasoning:', 'Mean > median > mode indicates a right-skewed distribution with a long upper tail.')

Q1 = clean_df['fare'].quantile(0.25)
Q3 = clean_df['fare'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
fare_outliers = clean_df[(clean_df['fare'] < lower) | (clean_df['fare'] > upper)]
print('Fare outliers:', len(fare_outliers))


In [ ]:
male_survival = clean_df.loc[clean_df['sex'] == 'male', 'survived'].mean()
female_survival = clean_df.loc[clean_df['sex'] == 'female', 'survived'].mean()
print('Male survival rate:', male_survival)
print('Female survival rate:', female_survival)

for pclass in sorted(clean_df['pclass'].unique()):
    rate = clean_df.loc[clean_df['pclass'] == pclass, 'survived'].mean()
    print(f'Pclass {pclass}: {rate:.3f}')

for sex in clean_df['sex'].dropna().unique():
    for pclass in sorted(clean_df['pclass'].unique()):
        mask = (clean_df['sex'] == sex) & (clean_df['pclass'] == pclass)
        rate = clean_df.loc[mask, 'survived'].mean()
        print(f'{sex}, Pclass {pclass}: {rate:.3f}')

plt.figure(figsize=(8, 5))
clean_df.groupby('sex')['survived'].mean().plot(kind='bar', color='#7aa6c2')
plt.title('Survival by sex')
plt.ylabel('Survival rate')
plt.tight_layout()
plt.savefig(out_dir / 'survival_by_sex.png', dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
clean_df.groupby('pclass')['survived'].mean().plot(kind='bar', color='#7aa6c2')
plt.title('Survival by pclass')
plt.ylabel('Survival rate')
plt.tight_layout()
plt.savefig(out_dir / 'survival_by_pclass.png', dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
sex_pclass = clean_df.groupby(['sex', 'pclass'])['survived'].mean().unstack()
sex_pclass.plot(kind='bar', figsize=(8, 5))
plt.title('Survival by sex and pclass')
plt.ylabel('Survival rate')
plt.tight_layout()
plt.savefig(out_dir / 'survival_by_sex_pclass.png', dpi=150)
plt.close()


In [ ]:
corr_columns = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr = clean_df[corr_columns].corr()
print(corr.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Titanic Correlation Matrix')
plt.tight_layout()
plt.savefig(out_dir / 'correlation_heatmap.png', dpi=150)
plt.close()

corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        col1 = corr.columns[i]
        col2 = corr.columns[j]
        value = corr.iloc[i, j]
        corr_pairs.append((col1, col2, value, abs(value)))

print('Two strongest correlations:')
for pair in sorted(corr_pairs, key=lambda x: x[3], reverse=True)[:2]:
    print(pair)


### Interpretation
The survival-by-sex chart shows a strong female advantage, which means sex is strongly associated with survival odds. The passenger-class chart reveals a class gradient, indicating that social position strongly influenced the chance of survival. The sex-by-class chart makes this even clearer: women in first class survived far more often than men in third class. These patterns suggest that both social status and gender helped shape survival outcomes on the ship.


In [ ]:
clean_df['age_z'] = (clean_df['age'] - clean_df['age'].mean()) / clean_df['age'].std()
clean_df['fare_z'] = (clean_df['fare'] - clean_df['fare'].mean()) / clean_df['fare'].std()
print('Age standardized mean:', clean_df['age_z'].mean())
print('Age standardized std:', clean_df['age_z'].std())
print('Fare standardized mean:', clean_df['fare_z'].mean())
print('Fare standardized std:', clean_df['fare_z'].std())
